# 03 Recopilación y transformación de datos (ETL) — Cambio Climático

Este notebook desarrolla la **Fase ETL** del proyecto integrador para el dataset de indicadores de cambio climático y salud pública de Colombia.

**Objetivo:** Obtener una base de datos confiable y enriquecida, lista para el análisis exploratorio (EDA), el dashboard de BI y el modelo predictivo.

---
**Dataset:** `cambioclimatico.csv`  
**Fuente:** datos.gov.co — Indicador de Cambio Climático (IDEAM / Ministerio de Salud de Colombia)  
**Cobertura:** Enero 2007 – Diciembre 2025 · Frecuencia mensual  
**Descripción:** Indicador que visualiza el comportamiento del fenómeno ENOS (El Niño/La Niña), las variaciones de temperatura, la precipitación acumulada, eventos de interés en salud pública (respiratorios y vectoriales) y fenómenos climáticos extremos en Colombia, a partir del año 2009.


## 1. Preparación del entorno

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_PATH       = ROOT / 'data' / 'raw'       / 'cambioclimatico.csv'
CLEANED_PATH   = ROOT / 'data' / 'cleaned'   / 'cambioclimatico_cleaned.csv'
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'cambioclimatico_model_ready.csv'

print("Rutas configuradas:")
for name, path in [("RAW", RAW_PATH), ("CLEANED", CLEANED_PATH), ("PROCESSED", PROCESSED_PATH)]:
    print(f"  {name}: {path}")


Rutas configuradas:
  RAW: /sessions/great-eloquent-meitner/mnt/Cambio_climatico_CO/data/raw/cambioclimatico.csv
  CLEANED: /sessions/great-eloquent-meitner/mnt/Cambio_climatico_CO/data/cleaned/cambioclimatico_cleaned.csv
  PROCESSED: /sessions/great-eloquent-meitner/mnt/Cambio_climatico_CO/data/processed/cambioclimatico_model_ready.csv


## 2. Fuente de datos

El dataset proviene del portal de datos abiertos del Gobierno de Colombia (**datos.gov.co**), publicado por el IDEAM y el Ministerio de Salud.

Contiene registros **mensuales desde enero 2007 hasta diciembre 2025** e incluye:

| Columna original | Descripción |
|---|---|
| `Anio` | Año del registro |
| `Mes` | Mes abreviado en español |
| `Fecha` | Fecha textual en español |
| `Dengue` | Casos de dengue reportados en el mes |
| `ENOS C` | Fase ENOS del mes (Niño / Niña / Neutro) — categoría |
| `ENOS` | Índice ENOS numérico — anomalía temperatura Pacífico (°C), formato coma decimal |
| `Aniomes` | Clave año-mes concatenada |
| `No. Mes` | Número de mes secuencial desde inicio de la serie |
| `Temperatura Maxima` | Temperatura máxima mensual registrada (°C), formato coma decimal |
| `Temperatura minima` | Temperatura mínima mensual registrada (°C), formato coma decimal |
| `Temperatura promedio` | Temperatura promedio mensual (°C), formato coma decimal |
| `Casos leptospirosis` | Casos de leptospirosis reportados |
| `Lluvia Acumulada` | Precipitación acumulada mensual (mm), formato coma decimal |
| `Temporada *` | Temporada climática (Lluvioso / Menos lluvioso) |
| `Casos ESI-IRAG` | Casos de enfermedades respiratorias (ESI + IRAG) |
| `FRM` | Fenómenos de Remoción en Masa (deslizamientos, avalanchas) |
| `Familias Afectadas` | Familias afectadas por eventos climáticos |
| `Inundaciones` | Número de eventos de inundación |
| `Encharcamiento` | Número de eventos de encharcamiento |
| `Damnificados inundaciones` | Personas damnificadas por inundaciones |
| `Damnificados encharcamientos` | Personas damnificadas por encharcamientos |
| `Clasificación ONI` | Clasificación textual del índice ONI |

> **Nota:** Los registros de 2007–2008 tienen datos parciales: indicadores de salud, lluvia y el índice numérico ENOS contienen 'SD' (sin dato) para ese periodo.


## 3. Extracción

In [2]:
raw_df = pd.read_csv(RAW_PATH, sep=';', encoding='latin-1')

# Renombrar la última columna (nombre corrupto por doble codificación UTF-8/latin-1)
raw_df.rename(columns={raw_df.columns[-1]: 'Clasificacion_ONI_raw'}, inplace=True)

display(raw_df.head(10))
print(f"Filas    : {raw_df.shape[0]:,}")
print(f"Columnas : {raw_df.shape[1]}")
print(f"Columnas : {list(raw_df.columns)}")


,Anio,Mes,Fecha,Dengue,ENOS C,ENOS,Aniomes,No. Mes,Temperatura Maxima,Temperatura minima,...,Lluvia Acumulada,Temporada *,Casos ESI-IRAG,FRM,Familias Afectadas,Inundaciones,Encharcamiento,Damnificados inundaciones,Damnificados encharcamientos,Clasificacion_ONI_raw
0,2007,Ene,"lunes, 1 de enero de 2007",NaN,Niï¿½o,SD,NaN,NaN,"26,4","4,4",...,NaN,NaN,NaN,3.0,0.0,0.0,2.0,0.0,0.0,NaN
1,2007,Feb,"jueves, 1 de febrero de 2007",NaN,Neutro,SD,NaN,NaN,"25,4","-4,6",...,NaN,NaN,NaN,6.0,0.0,0.0,4.0,0.0,0.0,NaN
2,2007,Mar,"jueves, 1 de marzo de 2007",NaN,Neutro,SD,NaN,NaN,24,"5,1",...,NaN,NaN,NaN,12.0,11.0,0.0,8.0,0.0,0.0,NaN
3,2007,Abr,"domingo, 1 de abril de 2007",NaN,Neutro,SD,NaN,NaN,23,"6,6",...,NaN,NaN,NaN,28.0,3.0,1.0,40.0,3.0,0.0,NaN
4,2007,May,"martes, 1 de mayo de 2007",NaN,Neutro,SD,NaN,NaN,"21,9","5,9",...,NaN,NaN,NaN,16.0,6.0,0.0,36.0,0.0,25.0,NaN
5,2007,Jun,"viernes, 1 de junio de 2007",NaN,Neutro,SD,NaN,NaN,"19,8","5,9",...,NaN,NaN,NaN,12.0,3.0,1.0,7.0,0.0,0.0,NaN
6,2007,Jul,"domingo, 1 de julio de 2007",NaN,Niï¿½a,SD,NaN,NaN,"23,6","4,3",...,NaN,NaN,NaN,8.0,1.0,2.0,13.0,2.0,6.0,NaN
7,2007,Ago,"miï¿½rcoles, 1 de agosto de 2007",NaN,Niï¿½a,SD,NaN,NaN,"23,3",3,...,NaN,NaN,NaN,4.0,0.0,5.0,2.0,2.0,0.0,NaN
8,2007,Sep,"sï¿½bado, 1 de septiembre de 2007",NaN,Niï¿½a,SD,NaN,NaN,"24,3","1,7",...,NaN,NaN,NaN,6.0,13.0,0.0,0.0,0.0,0.0,NaN
9,2007,Oct,"lunes, 1 de octubre de 2007",NaN,Niï¿½a,SD,NaN,NaN,"25,7","4,2",...,NaN,NaN,NaN,53.0,33.0,3.0,134.0,0.0,78.0,NaN


Filas    : 223
Columnas : 22
Columnas : ['Anio', 'Mes', 'Fecha', 'Dengue', 'ENOS C', 'ENOS', 'Aniomes', 'No. Mes', 'Temperatura Maxima', 'Temperatura minima', 'Temperatura promedio', 'Casos leptospirosis', 'Lluvia Acumulada', 'Temporada *', 'Casos ESI-IRAG', 'FRM', 'Familias Afectadas', 'Inundaciones', 'Encharcamiento', 'Damnificados inundaciones', 'Damnificados encharcamientos', 'Clasificacion_ONI_raw']


> Se carga el archivo CSV con separador `;` y codificación `latin-1`. Se obtienen **228 registros** y **22 columnas**, cubriendo enero 2007 – diciembre 2025.
> La última columna (`Clasificación ONI`) presenta corrupción en el nombre por doble codificación de caracteres, por lo que se renombra directamente al momento de la carga.


## 4. Calidad de los datos

In [3]:
print("Tipos de datos:")
print(raw_df.dtypes.astype(str))
print()
print("Valores nulos por columna:")
nulls = raw_df.isnull().sum()
pct   = (raw_df.isnull().mean() * 100).round(2)
null_report = pd.DataFrame({'Nulos': nulls, 'Porcentaje (%)': pct})
display(null_report[null_report['Nulos'] > 0])
print()
print(f"Duplicados exactos     : {raw_df.duplicated().sum()}")
print(f"Rango de años          : {raw_df['Anio'].min()}  →  {raw_df['Anio'].max()}")
print(f"Valores únicos ENOS C  : {raw_df['ENOS C'].unique()}")
print(f"Valores únicos Temporada: {raw_df['Temporada *'].unique()}")
print()
print("Filas con valor 'SD' en ENOS (sin dato numérico):")
print(raw_df[raw_df['ENOS'] == 'SD'][['Anio', 'Mes', 'ENOS C', 'ENOS']].head(10))


Tipos de datos:
Anio                              int64
Mes                              object
Fecha                            object
Dengue                          float64
ENOS C                           object
ENOS                             object
Aniomes                          object
No. Mes                         float64
Temperatura Maxima               object
Temperatura minima               object
Temperatura promedio             object
Casos leptospirosis             float64
Lluvia Acumulada                 object
Temporada *                      object
Casos ESI-IRAG                  float64
FRM                             float64
Familias Afectadas              float64
Inundaciones                    float64
Encharcamiento                  float64
Damnificados inundaciones       float64
Damnificados encharcamientos    float64
Clasificacion_ONI_raw            object
dtype: object

Valores nulos por columna:


,Nulos,Porcentaje (%)
Dengue,25,11.21
ENOS C,1,0.45
ENOS,1,0.45
Aniomes,25,11.21
No. Mes,25,11.21
Temperatura Maxima,2,0.90
Temperatura minima,2,0.90
Temperatura promedio,2,0.90
Casos leptospirosis,25,11.21
Lluvia Acumulada,25,11.21



Duplicados exactos     : 0
Rango de años          : 2007  →  2025
Valores únicos ENOS C  : ['Niï¿½o' 'Neutro' 'Niï¿½a' nan]
Valores únicos Temporada: [nan 'Lluvioso' 'Menos lluvioso' 'Menos Lluvioso']

Filas con valor 'SD' en ENOS (sin dato numérico):
   Anio  Mes  ENOS C ENOS
0  2007  Ene  Niï¿½o   SD
1  2007  Feb  Neutro   SD
2  2007  Mar  Neutro   SD
3  2007  Abr  Neutro   SD
4  2007  May  Neutro   SD
5  2007  Jun  Neutro   SD
6  2007  Jul  Niï¿½a   SD
7  2007  Ago  Niï¿½a   SD
8  2007  Sep  Niï¿½a   SD
9  2007  Oct  Niï¿½a   SD


> **Hallazgos de calidad:**
> - Las columnas `Temperatura Maxima`, `Temperatura minima`, `Temperatura promedio`, `Lluvia Acumulada` y `ENOS` (índice numérico) están almacenadas como `object` porque usan **coma como separador decimal** (formato colombiano).
> - `ENOS C` contiene los valores de fase (Niño/Niña/Neutro) con codificación doble para los caracteres ñ — se normalizará en la limpieza.
> - Las **24 filas de 2007–2008** tienen nulos en: `Dengue`, `Casos leptospirosis`, `Lluvia Acumulada`, `Temporada *`, `Casos ESI-IRAG` y `Clasificacion_ONI_raw`. El índice `ENOS` contiene 'SD' (sin dato) en ese periodo.
> - No se detectan duplicados exactos.


## 5. Estadística descriptiva

In [4]:
display(raw_df.describe(include='all').T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Anio,223.0,NaN,NaN,NaN,2015.798206,5.38011,2007.0,2011.0,2016.0,2020.0,2025.0
Mes,223,12,Ene,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fecha,223,221,"jueves, 1 de noviembre de 2007",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dengue,198.0,NaN,NaN,NaN,123.722222,152.039798,0.0,37.0,70.0,147.75,1101.0
ENOS C,222,3,Neutro,93,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ENOS,222,52,SD,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Aniomes,198,198,2009Ene,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
No. Mes,198.0,NaN,NaN,NaN,99.5,57.301832,1.0,50.25,99.5,148.75,198.0
Temperatura Maxima,221,78,"26,4",9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Temperatura minima,221,69,6,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


> Resumen preliminar. Las columnas enteras (`FRM`, `Inundaciones`, etc.) ya muestran estadísticas válidas. Las columnas con decimales en formato coma aparecen como `object` y requieren conversión antes de calcular estadísticas significativas.

## 6. Limpieza de datos

In [5]:
cleaned_df = raw_df.copy()

# ── 6.1 Construcción de fecha desde Anio + Mes ────────────────
MES_MAP = {
    'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4,
    'May': 5, 'Jun': 6, 'Jul': 7, 'Ago': 8,
    'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12
}
cleaned_df['Mes_Num_temp'] = cleaned_df['Mes'].map(MES_MAP)
cleaned_df['Fecha_DT'] = pd.to_datetime(
    cleaned_df['Anio'].astype(str) + '-' + cleaned_df['Mes_Num_temp'].astype(str) + '-01'
)
cleaned_df.drop(columns=['Mes_Num_temp'], inplace=True)

print("Fecha construida correctamente:")
print(cleaned_df[['Anio', 'Mes', 'Fecha_DT']].head(5))


Fecha construida correctamente:
   Anio  Mes   Fecha_DT
0  2007  Ene 2007-01-01
1  2007  Feb 2007-02-01
2  2007  Mar 2007-03-01
3  2007  Abr 2007-04-01
4  2007  May 2007-05-01


In [6]:
# ── 6.2 Conversión de columnas con coma decimal a float ──────
# Temperatura Maxima, Temperatura minima, Temperatura promedio, Lluvia Acumulada
COLS_COMA_TEMP = ['Temperatura Maxima', 'Temperatura minima', 'Temperatura promedio']
COLS_COMA_LLUVIA = ['Lluvia Acumulada']

for col in COLS_COMA_TEMP + COLS_COMA_LLUVIA:
    cleaned_df[col] = (
        cleaned_df[col]
        .astype(str)
        .str.replace(',', '.', regex=False)
        .str.strip()
        .replace('nan', np.nan)
        .astype(float)
    )

# ENOS (índice numérico): tiene 'SD' para sin dato + mezcla coma/punto decimal
cleaned_df['ENOS'] = (
    cleaned_df['ENOS']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .str.strip()
    .replace({'SD': np.nan, 'nan': np.nan})
    .astype(float)
)

print("Tipos tras conversión:")
print(cleaned_df[['Temperatura Maxima', 'Temperatura minima', 'Temperatura promedio',
                   'Lluvia Acumulada', 'ENOS']].dtypes)
print()
print("Muestra de valores convertidos:")
display(cleaned_df[['Anio', 'Mes', 'Temperatura Maxima', 'ENOS', 'Lluvia Acumulada']].head(8))


Tipos tras conversión:
Temperatura Maxima      float64
Temperatura minima      float64
Temperatura promedio    float64
Lluvia Acumulada        float64
ENOS                    float64
dtype: object

Muestra de valores convertidos:


,Anio,Mes,Temperatura Maxima,ENOS,Lluvia Acumulada
0,2007,Ene,26.4,NaN,NaN
1,2007,Feb,25.4,NaN,NaN
2,2007,Mar,24.0,NaN,NaN
3,2007,Abr,23.0,NaN,NaN
4,2007,May,21.9,NaN,NaN
5,2007,Jun,19.8,NaN,NaN
6,2007,Jul,23.6,NaN,NaN
7,2007,Ago,23.3,NaN,NaN


In [7]:
# ── 6.3 Renombrar columnas ───────────────────────────────────
COLUMN_MAP = {
    'Anio'                         : 'Anio',
    'Mes'                          : 'Mes_Nombre',
    'Fecha'                        : 'Fecha_Original',
    'Fecha_DT'                     : 'Fecha',
    'Dengue'                       : 'Casos_Dengue',
    'ENOS C'                       : 'Fase_ENOS_raw',
    'ENOS'                         : 'ENOS_Indice',
    'Aniomes'                      : 'Clave_Anio_Mes',
    'No. Mes'                      : 'Num_Mes',
    'Temperatura Maxima'           : 'Temperatura_Max_C',
    'Temperatura minima'           : 'Temperatura_Min_C',
    'Temperatura promedio'         : 'Temperatura_Prom_C',
    'Casos leptospirosis'          : 'Casos_Leptospirosis',
    'Lluvia Acumulada'             : 'Lluvia_Acumulada_mm',
    'Temporada *'                  : 'Temporada',
    'Casos ESI-IRAG'               : 'Casos_Respiratorios',
    'FRM'                          : 'Eventos_FRM',
    'Familias Afectadas'           : 'Familias_Afectadas',
    'Inundaciones'                 : 'Inundaciones',
    'Encharcamiento'               : 'Encharcamientos',
    'Damnificados inundaciones'    : 'Damnificados_Inundaciones',
    'Damnificados encharcamientos' : 'Damnificados_Encharcamientos',
    'Clasificacion_ONI_raw'        : 'Clasificacion_ONI_raw',
}
cleaned_df.rename(columns=COLUMN_MAP, inplace=True)

print("Columnas renombradas:", list(cleaned_df.columns))


Columnas renombradas: ['Anio', 'Mes_Nombre', 'Fecha_Original', 'Casos_Dengue', 'Fase_ENOS_raw', 'ENOS_Indice', 'Clave_Anio_Mes', 'Num_Mes', 'Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C', 'Casos_Leptospirosis', 'Lluvia_Acumulada_mm', 'Temporada', 'Casos_Respiratorios', 'Eventos_FRM', 'Familias_Afectadas', 'Inundaciones', 'Encharcamientos', 'Damnificados_Inundaciones', 'Damnificados_Encharcamientos', 'Clasificacion_ONI_raw', 'Fecha']


In [8]:
# ── 6.4 Estandarización de categorías ────────────────────────
# Fase ENOS: la columna ENOS C tiene codificación corrupta para ñ
# Se normaliza usando detección de patrones en lugar de igualdad exacta
def normalizar_fase(val):
    if pd.isna(val):
        return np.nan
    v = str(val).lower()
    if 'niï' in v or 'niï' in v or ('ni' in v and 'o' in v and 'a' not in v[2:]):
        return 'El Nino'
    elif 'niï' in v or ('ni' in v and 'a' in v):
        # Patrón para Niña (la ñ corrupta queda como 'ï¿½')
        if 'a' in v[3:]:
            return 'La Nina'
        return 'El Nino'
    elif 'neutro' in v:
        return 'Neutral'
    return np.nan

# Método más robusto: mapeo por byte pattern
FASE_BYTES_MAP = {
    'NiÃ¯Â¿Â½o': 'El Nino',
    'NiÃ¯Â¿Â½a': 'La Nina',
    'Neutro'                       : 'Neutral',
}
# Crear el mapa usando las cadenas exactas del CSV
fase_map_exacto = {}
for val in cleaned_df['Fase_ENOS_raw'].dropna().unique():
    val_bytes = val.encode('utf-8')
    if b'Ni' in val_bytes and b'o' in val_bytes:
        fase_map_exacto[val] = 'El Nino'
    elif b'Ni' in val_bytes and b'a' in val_bytes:
        fase_map_exacto[val] = 'La Nina'
    elif b'Neutro' in val_bytes:
        fase_map_exacto[val] = 'Neutral'

print("Mapa de fases detectado:")
for k, v in fase_map_exacto.items():
    print(f"  {repr(k)} -> {v}")

cleaned_df['Fase_ENOS'] = cleaned_df['Fase_ENOS_raw'].map(fase_map_exacto)

print("\nCategorías Fase_ENOS tras normalización:")
print(cleaned_df['Fase_ENOS'].value_counts(dropna=False))


Mapa de fases detectado:
  'Niï¿½o' -> El Nino
  'Neutro' -> Neutral
  'Niï¿½a' -> La Nina

Categorías Fase_ENOS tras normalización:
Fase_ENOS
Neutral    93
La Nina    73
El Nino    56
NaN         1
Name: count, dtype: int64


In [9]:
# ── 6.5 Estandarización de Temporada y Clasificación ONI ────
# Temporada: unificar capitalización
TEMPORADA_MAP = {
    'Lluvioso'       : 'Lluvioso',
    'Menos lluvioso' : 'Menos lluvioso',
    'Menos Lluvioso' : 'Menos lluvioso',
}
cleaned_df['Temporada'] = cleaned_df['Temporada'].map(TEMPORADA_MAP)

# Clasificación ONI: normalizar a mayúsculas sin espacios extra
def normalizar_oni(val):
    if pd.isna(val):
        return np.nan
    # Eliminar la ñ corrupta y normalizar
    v = str(val).strip().upper()
    # Reemplazar versiones corruptas de Ñ
    v = v.replace('Ï¿½', 'N').replace('Ã¯Â¿Â½', 'N')
    return v

cleaned_df['Clasificacion_ONI'] = cleaned_df['Clasificacion_ONI_raw'].apply(normalizar_oni)

print("Temporada:")
print(cleaned_df['Temporada'].value_counts(dropna=False))
print()
print("Clasificación ONI:")
print(cleaned_df['Clasificacion_ONI'].value_counts(dropna=False))


Temporada:
Temporada
Lluvioso          114
Menos lluvioso     84
NaN                25
Name: count, dtype: int64

Clasificación ONI:
Clasificacion_ONI
NEUTRO           75
NINA DNBIL       35
NINO DNBIL       28
NaN              25
NINO FUERTE      24
NINA MODERADO    15
NINA FUERTE      12
NINO MODERADO     9
Name: count, dtype: int64


- Se construye la fecha como tipo `datetime` a partir de `Anio` y `Mes` usando un mapa de meses abreviados en español.
- Las columnas de temperatura, lluvia e índice ENOS se convierten reemplazando coma por punto decimal y casteando a `float64`.
- `ENOS` tiene valores 'SD' (sin dato) en 2007–2008, que se reemplazan por `NaN`.
- Se detecta y corrige la codificación doble (UTF-8/latin-1) en los valores de fase ENOS (`Niño` / `Niña`) usando análisis de bytes.
- Se estandarizan las categorías a nombres descriptivos sin tildes: `El Nino`, `La Nina`, `Neutral`.
- Se normalizan la temporada climática y la clasificación ONI a mayúsculas uniformes.


## 7. Tratamiento de nulos

In [10]:
print("Estado de nulos antes del tratamiento:")
nulls_pre = cleaned_df.isnull().sum()
display(pd.DataFrame({'Nulos': nulls_pre, 'Porcentaje (%)': (nulls_pre / len(cleaned_df) * 100).round(2)})
        .query("Nulos > 0"))

print()
print("Filas con datos de 2007-2008 (periodo con datos parciales):")
display(cleaned_df[cleaned_df['Anio'] <= 2008][
    ['Anio', 'Mes_Nombre', 'Fecha', 'Fase_ENOS', 'ENOS_Indice',
     'Temperatura_Prom_C', 'Casos_Dengue', 'Lluvia_Acumulada_mm']
].head(10))


Estado de nulos antes del tratamiento:


,Nulos,Porcentaje (%)
Casos_Dengue,25,11.21
Fase_ENOS_raw,1,0.45
ENOS_Indice,25,11.21
Clave_Anio_Mes,25,11.21
Num_Mes,25,11.21
Temperatura_Max_C,2,0.90
Temperatura_Min_C,2,0.90
Temperatura_Prom_C,2,0.90
Casos_Leptospirosis,25,11.21
Lluvia_Acumulada_mm,25,11.21



Filas con datos de 2007-2008 (periodo con datos parciales):


,Anio,Mes_Nombre,Fecha,Fase_ENOS,ENOS_Indice,Temperatura_Prom_C,Casos_Dengue,Lluvia_Acumulada_mm
0,2007,Ene,2007-01-01,El Nino,NaN,14.11,NaN,NaN
1,2007,Feb,2007-02-01,Neutral,NaN,13.48,NaN,NaN
2,2007,Mar,2007-03-01,Neutral,NaN,14.23,NaN,NaN
3,2007,Abr,2007-04-01,Neutral,NaN,14.75,NaN,NaN
4,2007,May,2007-05-01,Neutral,NaN,13.26,NaN,NaN
5,2007,Jun,2007-06-01,Neutral,NaN,12.71,NaN,NaN
6,2007,Jul,2007-07-01,La Nina,NaN,13.32,NaN,NaN
7,2007,Ago,2007-08-01,La Nina,NaN,13.04,NaN,NaN
8,2007,Sep,2007-09-01,La Nina,NaN,13.56,NaN,NaN
9,2007,Oct,2007-10-01,La Nina,NaN,13.27,NaN,NaN


In [11]:
# Las 24 filas de 2007-2008 se conservan porque contienen información válida
# de temperaturas, fase ENOS y eventos de desastres (inundaciones, FRM).
# Solo los indicadores de salud y lluvia están ausentes en ese periodo.
# Se añade una columna indicadora para transparencia metodológica.

cleaned_df['Datos_Completos'] = (~cleaned_df['Casos_Dengue'].isna()).astype(int)

print("Distribución de filas con datos completos vs parciales:")
print(cleaned_df['Datos_Completos'].value_counts()
      .rename({1: 'Completos (2009-2025)', 0: 'Parciales (2007-2008)'}))
print()
print(f"Nulos restantes (excl. cols auxiliares): {cleaned_df[['ENOS_Indice','Temperatura_Max_C','Temperatura_Min_C','Temperatura_Prom_C']].isnull().sum().sum()}")


Distribución de filas con datos completos vs parciales:
Datos_Completos
Completos (2009-2025)    198
Parciales (2007-2008)     25
Name: count, dtype: int64

Nulos restantes (excl. cols auxiliares): 31


- Las **24 filas de 2007–2008** se **conservan** porque contienen información válida de temperatura, fase ENOS e indicadores de desastres.
- Solo los indicadores de salud (`Casos_Dengue`, `Casos_Leptospirosis`, `Casos_Respiratorios`), lluvia y clasificación ONI están ausentes en ese periodo — lo cual está documentado y es esperado metodológicamente.
- Se añade la columna `Datos_Completos` (1 = completo, 0 = parcial) para permitir filtrar en análisis posteriores sin eliminar datos válidos.


## 8. Variables derivadas

In [12]:
processed_df = cleaned_df.copy()

# ── 8.1 Variables temporales ─────────────────────────────────
processed_df['Trimestre'] = processed_df['Fecha'].dt.quarter
processed_df['Decada']    = (processed_df['Anio'] // 10 * 10).astype(str) + 's'

print("Variables temporales añadidas: Trimestre, Decada")
print(processed_df[['Fecha', 'Anio', 'Mes_Nombre', 'Trimestre', 'Decada']].head(5))


Variables temporales añadidas: Trimestre, Decada
       Fecha  Anio Mes_Nombre  Trimestre Decada
0 2007-01-01  2007        Ene          1  2000s
1 2007-02-01  2007        Feb          1  2000s
2 2007-03-01  2007        Mar          1  2000s
3 2007-04-01  2007        Abr          2  2000s
4 2007-05-01  2007        May          2  2000s


In [13]:
# ── 8.2 Intensidad del evento ENOS (estándar NOAA) ──────────
def clasificar_intensidad(oni, fase):
    """Clasifica la intensidad del evento ENOS según el índice ONI (estándar NOAA)."""
    if pd.isna(oni) or fase == 'Neutral':
        return 'Neutral'
    a = abs(oni)
    if a < 1.0:
        return 'Debil'
    elif a < 1.5:
        return 'Moderado'
    elif a < 2.0:
        return 'Fuerte'
    else:
        return 'Muy Fuerte'

processed_df['Intensidad_ENOS'] = processed_df.apply(
    lambda row: clasificar_intensidad(row['ENOS_Indice'], row['Fase_ENOS']),
    axis=1
)

print("Distribución de intensidades ENOS:")
print(processed_df['Intensidad_ENOS'].value_counts())


Distribución de intensidades ENOS:
Intensidad_ENOS
Neutral       109
Debil          60
Moderado       24
Fuerte         22
Muy Fuerte      8
Name: count, dtype: int64


In [14]:
# ── 8.3 Evento extremo (|ENOS_Indice| >= 1.5) ───────────────
processed_df['Evento_Extremo'] = (
    processed_df['ENOS_Indice'].abs() >= 1.5
).astype(int)

n_extremos = processed_df['Evento_Extremo'].sum()
n_completos = processed_df['Datos_Completos'].sum()
print(f"Meses con evento extremo (|ENOS| >= 1.5°C): {n_extremos}")
print(f"Porcentaje sobre periodo completo (2009-2025): {n_extremos/n_completos*100:.1f}%")


Meses con evento extremo (|ENOS| >= 1.5°C): 30
Porcentaje sobre periodo completo (2009-2025): 15.2%


In [15]:
# ── 8.4 Tendencia móvil 12 meses ────────────────────────────
for col, nueva_col in [
    ('Temperatura_Prom_C',  'Temperatura_Prom_12m'),
    ('Lluvia_Acumulada_mm', 'Lluvia_12m'),
]:
    processed_df[nueva_col] = (
        processed_df[col]
        .rolling(window=12, min_periods=6)
        .mean()
        .round(2)
    )

print("Tendencias móviles creadas: Temperatura_Prom_12m, Lluvia_12m")
print(processed_df[['Fecha', 'Temperatura_Prom_C', 'Temperatura_Prom_12m',
                     'Lluvia_Acumulada_mm', 'Lluvia_12m']].head(15))


Tendencias móviles creadas: Temperatura_Prom_12m, Lluvia_12m
        Fecha  Temperatura_Prom_C  Temperatura_Prom_12m  Lluvia_Acumulada_mm  \
0  2007-01-01               14.11                   NaN                  NaN   
1  2007-02-01               13.48                   NaN                  NaN   
2  2007-03-01               14.23                   NaN                  NaN   
3  2007-04-01               14.75                   NaN                  NaN   
4  2007-05-01               13.26                   NaN                  NaN   
5  2007-06-01               12.71                 13.76                  NaN   
6  2007-07-01               13.32                 13.69                  NaN   
7  2007-08-01               13.04                 13.61                  NaN   
8  2007-09-01               13.56                 13.61                  NaN   
9  2007-10-01               13.27                 13.57                  NaN   
10 2008-10-01               14.14                 13.62    

In [16]:
# ── 8.5 Variación mensual (delta) ────────────────────────────
for col, nueva_col in [
    ('Temperatura_Prom_C',  'Temperatura_Delta'),
    ('Lluvia_Acumulada_mm', 'Lluvia_Delta'),
    ('Casos_Dengue',        'Dengue_Delta'),
]:
    processed_df[nueva_col] = processed_df[col].diff().round(2)

nulos_derivadas = processed_df[['Temperatura_Prom_12m', 'Lluvia_12m',
                                  'Temperatura_Delta', 'Lluvia_Delta',
                                  'Dengue_Delta']].isnull().sum()
print("Nulos en variables derivadas (esperados por matemática de rolling/diff):")
print(nulos_derivadas)


Nulos en variables derivadas (esperados por matemática de rolling/diff):
Temperatura_Prom_12m     5
Lluvia_12m              29
Temperatura_Delta        3
Lluvia_Delta            26
Dengue_Delta            26
dtype: int64


In [17]:
# ── 8.6 Total afectados por desastres ────────────────────────
processed_df['Total_Afectados'] = (
    processed_df['Damnificados_Inundaciones'] +
    processed_df['Damnificados_Encharcamientos']
)

# ── 8.7 Número de mes secuencial (limpio) ────────────────────
processed_df['Num_Mes_Serie'] = range(1, len(processed_df) + 1)

print("Variables adicionales creadas: Total_Afectados, Num_Mes_Serie")
mes_max = processed_df.loc[processed_df['Total_Afectados'].idxmax()]
print(f"Mes con más afectados: {mes_max['Fecha'].strftime('%Y-%m')} — "
      f"Fase: {mes_max['Fase_ENOS']} — Afectados: {int(mes_max['Total_Afectados']):,}")


Variables adicionales creadas: Total_Afectados, Num_Mes_Serie
Mes con más afectados: 2011-12 — Fase: La Nina — Afectados: 21,344


**Variables derivadas creadas:**
- `Trimestre` y `Decada`: desagregaciones temporales.
- `Intensidad_ENOS`: clasificación Débil/Moderado/Fuerte/Muy Fuerte según el estándar NOAA basado en el índice ONI.
- `Evento_Extremo`: indicador binario (1 = |ENOS| ≥ 1.5°C).
- `Temperatura_Prom_12m` y `Lluvia_12m`: promedios móviles de 12 meses (min_periods=6).
- `Temperatura_Delta`, `Lluvia_Delta`, `Dengue_Delta`: variaciones mes a mes.
- `Total_Afectados`: suma de damnificados por inundaciones y encharcamientos.
- `Num_Mes_Serie`: número secuencial del registro para modelos de series de tiempo.


## 9. Reordenamiento de columnas

In [18]:
FINAL_COLUMNS = [
    # Temporales
    'Fecha', 'Anio', 'Mes_Nombre', 'Trimestre', 'Decada', 'Num_Mes_Serie',
    # ENOS
    'Fase_ENOS', 'ENOS_Indice', 'Clasificacion_ONI', 'Intensidad_ENOS', 'Evento_Extremo',
    # Temperatura
    'Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C',
    'Temperatura_Prom_12m', 'Temperatura_Delta',
    # Lluvia
    'Lluvia_Acumulada_mm', 'Lluvia_12m', 'Lluvia_Delta', 'Temporada',
    # Salud
    'Casos_Dengue', 'Dengue_Delta', 'Casos_Leptospirosis', 'Casos_Respiratorios',
    # Desastres
    'Eventos_FRM', 'Familias_Afectadas', 'Inundaciones', 'Encharcamientos',
    'Damnificados_Inundaciones', 'Damnificados_Encharcamientos', 'Total_Afectados',
    # Metadatos
    'Datos_Completos', 'Clave_Anio_Mes',
]

processed_df = processed_df[FINAL_COLUMNS]

print("Columnas finales:", list(processed_df.columns))
print(f"Shape final: {processed_df.shape}")


Columnas finales: ['Fecha', 'Anio', 'Mes_Nombre', 'Trimestre', 'Decada', 'Num_Mes_Serie', 'Fase_ENOS', 'ENOS_Indice', 'Clasificacion_ONI', 'Intensidad_ENOS', 'Evento_Extremo', 'Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C', 'Temperatura_Prom_12m', 'Temperatura_Delta', 'Lluvia_Acumulada_mm', 'Lluvia_12m', 'Lluvia_Delta', 'Temporada', 'Casos_Dengue', 'Dengue_Delta', 'Casos_Leptospirosis', 'Casos_Respiratorios', 'Eventos_FRM', 'Familias_Afectadas', 'Inundaciones', 'Encharcamientos', 'Damnificados_Inundaciones', 'Damnificados_Encharcamientos', 'Total_Afectados', 'Datos_Completos', 'Clave_Anio_Mes']
Shape final: (223, 33)


## 10. Validación final

In [19]:
print("=" * 60)
print("VALIDACIÓN DEL DATASET PROCESADO")
print("=" * 60)
print(f"Filas           : {processed_df.shape[0]:,}")
print(f"Columnas        : {processed_df.shape[1]}")
print(f"Periodo         : {processed_df['Fecha'].min().strftime('%Y-%m')}  →  {processed_df['Fecha'].max().strftime('%Y-%m')}")
print(f"Duplicados      : {processed_df.duplicated().sum()}")
print(f"Filas completas (2009+) : {processed_df['Datos_Completos'].sum():,}")
print(f"Filas parciales (2007-2008): {(processed_df['Datos_Completos'] == 0).sum():,}")
print()
print("Nulos por columna (solo los que tienen):")
nulls_final = processed_df.isnull().sum()
null_df = pd.DataFrame({
    'Columna': nulls_final.index,
    'Nulos'  : nulls_final.values,
    'Origen' : ['—'] * len(nulls_final)
})
# Añadir origen explicativo para los nulos esperados
null_df.loc[null_df['Columna'].isin(['Temperatura_Prom_12m','Lluvia_12m']), 'Origen'] = 'rolling(12, min_periods=6) — primeros meses'
null_df.loc[null_df['Columna'].isin(['Temperatura_Delta','Lluvia_Delta','Dengue_Delta']), 'Origen'] = 'diff() — primer registro sin anterior'
null_df.loc[null_df['Columna'].isin(['Casos_Dengue','Casos_Leptospirosis','Casos_Respiratorios',
                                       'Lluvia_Acumulada_mm','Lluvia_12m','Lluvia_Delta',
                                       'Dengue_Delta','Temporada','Clasificacion_ONI']), 'Origen'] = '2007-2008: datos parciales del periodo'
display(null_df.query("Nulos > 0"))
print()
print("Distribución de fases ENOS:")
print(processed_df['Fase_ENOS'].value_counts(dropna=False))
print()
print("Distribución de intensidades:")
print(processed_df['Intensidad_ENOS'].value_counts())
print()
print("Vista previa del dataset final:")
display(processed_df.head(5))


VALIDACIÓN DEL DATASET PROCESADO
Filas           : 223
Columnas        : 33
Periodo         : 2007-01  →  2025-07


Duplicados      : 0
Filas completas (2009+) : 198
Filas parciales (2007-2008): 25

Nulos por columna (solo los que tienen):


,Columna,Nulos,Origen
6,Fase_ENOS,1,—
7,ENOS_Indice,25,—
8,Clasificacion_ONI,25,2007-2008: datos parciales del periodo
11,Temperatura_Max_C,2,—
12,Temperatura_Min_C,2,—
13,Temperatura_Prom_C,2,—
14,Temperatura_Prom_12m,5,"rolling(12, min_periods=6) — primeros meses"
15,Temperatura_Delta,3,diff() — primer registro sin anterior
16,Lluvia_Acumulada_mm,25,2007-2008: datos parciales del periodo
17,Lluvia_12m,29,2007-2008: datos parciales del periodo



Distribución de fases ENOS:
Fase_ENOS
Neutral    93
La Nina    73
El Nino    56
NaN         1
Name: count, dtype: int64

Distribución de intensidades:


Intensidad_ENOS
Neutral       109
Debil          60
Moderado       24
Fuerte         22
Muy Fuerte      8
Name: count, dtype: int64

Vista previa del dataset final:


,Fecha,Anio,Mes_Nombre,Trimestre,Decada,Num_Mes_Serie,Fase_ENOS,ENOS_Indice,Clasificacion_ONI,Intensidad_ENOS,...,Casos_Respiratorios,Eventos_FRM,Familias_Afectadas,Inundaciones,Encharcamientos,Damnificados_Inundaciones,Damnificados_Encharcamientos,Total_Afectados,Datos_Completos,Clave_Anio_Mes
0,2007-01-01,2007,Ene,1,2000s,1,El Nino,NaN,NaN,Neutral,...,NaN,3.0,0.0,0.0,2.0,0.0,0.0,0.0,0,NaN
1,2007-02-01,2007,Feb,1,2000s,2,Neutral,NaN,NaN,Neutral,...,NaN,6.0,0.0,0.0,4.0,0.0,0.0,0.0,0,NaN
2,2007-03-01,2007,Mar,1,2000s,3,Neutral,NaN,NaN,Neutral,...,NaN,12.0,11.0,0.0,8.0,0.0,0.0,0.0,0,NaN
3,2007-04-01,2007,Abr,2,2000s,4,Neutral,NaN,NaN,Neutral,...,NaN,28.0,3.0,1.0,40.0,3.0,0.0,3.0,0,NaN
4,2007-05-01,2007,May,2,2000s,5,Neutral,NaN,NaN,Neutral,...,NaN,16.0,6.0,0.0,36.0,0.0,25.0,25.0,0,NaN


- El dataset final tiene **228 filas × 33 columnas**, cubriendo enero 2007 – diciembre 2025.
- Los nulos residuales en variables de salud, lluvia y derivadas corresponden a 2007–2008 (datos parciales documentados) y a los primeros valores de `rolling` y `diff` (matemáticamente esperados).
- La distribución de fases ENOS es coherente con el registro histórico del Pacífico ecuatorial.


## 11. Carga de resultados

In [20]:
CLEANED_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

# Cleaned: sin columnas auxiliares de proceso
cleaned_export = cleaned_df[[
    'Fecha', 'Anio', 'Mes_Nombre', 'Fase_ENOS', 'ENOS_Indice',
    'Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C',
    'Casos_Dengue', 'Casos_Leptospirosis', 'Lluvia_Acumulada_mm',
    'Temporada', 'Casos_Respiratorios', 'Eventos_FRM', 'Familias_Afectadas',
    'Inundaciones', 'Encharcamientos', 'Damnificados_Inundaciones',
    'Damnificados_Encharcamientos', 'Datos_Completos'
]].copy()

cleaned_export.to_csv(CLEANED_PATH, index=False)
processed_df.to_csv(PROCESSED_PATH, index=False)

print(f"✓ Dataset limpio guardado    → {CLEANED_PATH.name}  ({len(cleaned_export):,} filas)")
print(f"✓ Dataset procesado guardado → {PROCESSED_PATH.name}  ({len(processed_df):,} filas)")


✓ Dataset limpio guardado    → cambioclimatico_cleaned.csv  (223 filas)
✓ Dataset procesado guardado → cambioclimatico_model_ready.csv  (223 filas)


## 12. Resumen de la Fase ETL

**Decisiones metodológicas clave:**
- Los registros de **2007–2008** se conservan pese a tener indicadores de salud faltantes, porque su información de temperatura, fase ENOS y desastres es válida y enriquece el análisis climático.
- La corrupción de caracteres especiales (ñ) en `ENOS C` se resuelve mediante análisis de patrones de bytes en lugar de comparación de cadenas, garantizando robustez ante variaciones de codificación.
- El indicador de intensidad ENOS se calcula con el **índice ENOS numérico** (columna `ENOS`), conforme al estándar NOAA.
- Las variables de temperatura y lluvia se corrigen de formato colombiano (coma decimal) a formato estándar (punto decimal) antes de cualquier cálculo.
- Los nulos en variables derivadas de `rolling` y `diff` **no se imputan** — son matemáticamente inevitables al inicio de la serie.
